# Import the dataset

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import linregress, ttest_ind
import matplotlib.cm as cm
from matplotlib.cm import ScalarMappable
import matplotlib.colors as mcolors

from shapely.geometry import Point
import statsmodels.formula.api as smf

**Reminder:** Our dataframe has 8352 data points and 32 variables per data point.

In [ ]:
df = pd.read_excel("clean_data_kp.xlsx")
df

In [ ]:
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

geometry = [Point(xy) for xy in zip(df["lon"], df["lat"])]
geo_rad = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

COLOR = "managua"

### Sub-Dataframes

In [ ]:
df_arctic = df[df['lat']>66.5].copy()
df_antarctic = df[df['lat']<-66.5].copy()

floored = geo_rad.copy()
floored[["kp", "kp_lag3", "kp_lag6", "kp_lag24", "kp_lag48"]] = np.floor(floored[["kp", "kp_lag3", "kp_lag6", "kp_lag24", "kp_lag48"]])

KP_MAX = 5.667
KP_MIN = 0

# KP Exploration

**KP signficiance:**

https://theaurorazone.com/nuts-about-kp/

|Kp Scale|Auroral Activity|Frequency|
|---|---|---|
|0|Quiet|
|1|Quiet|A range between Kp1 to Kp3|
|2|Quiet|is most frequently observed|
|3|Unsettled|
|4|Active|
|5|Minor Storm|900 Days per 11 Years|
|6|Moderate Storm|360 Days per 11 Years|
|7|Strong Storm|130 Days per 11 Years|
|8|Severe Storm|60 Days per 11 Years|
|9|Extreme Storm|4 Days per 11 Years|

In [ ]:
df.groupby("kp")["kp"].count()

In [ ]:
floored.groupby("kp")["kp"].count().plot(kind="bar")
plt.xlabel('Kp Index')
plt.ylabel('Counts')
plt.show()

## Preliminary Work

We decided to bin all of datapoints by flooring their Kp values. This created 6 bins ranging from 0 to 5 as all of the Kp values in our dataset ranged from 0.33 to 5.67. 

**Note:** At or above a Kp value of 5 is where there is considered to be a magnetic strom, and in our dataset we only have 123 datapoints out of 8352 datapoints that falls in this category.

In [ ]:
df["bin"] = np.floor(df["kp"]).astype(int).clip(0, 5)

In [ ]:
df['bin'].value_counts().sort_index()

In [ ]:
df_kp0 = df[df['bin']==0].copy()
df_kp1 = df[df['bin']==1].copy()
df_kp2 = df[df['bin']==2].copy()
df_kp3 = df[df['bin']==3].copy()
df_kp4 = df[df['bin']==4].copy()
df_kp5 = df[df['bin']==5].copy()

Next I wanted to get an idea as to the locations of the datapoints present in each of the 6 bins. This really makes apparent the lack of datapoints we have in the Kp 5 bin, but the other groups appear to have a good spread of datapoints from across the globe. 

### Heatmap Exploration

In [ ]:
fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6)) = plt.subplots(3, 2, figsize=(24, 18))

world.plot(ax=ax1, color="lightgray")
sc1 = ax1.scatter(
    df_kp0["lon"],
    df_kp0["lat"],
    c='black',
    s=5
)
ax1.set_title("kP=0 Locations")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")

world.plot(ax=ax2, color="lightgray")
sc2 = ax2.scatter(
    df_kp1["lon"],
    df_kp1["lat"],
    c='black',
    s=5
)
ax2.set_title("kP=1 Locations")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")

world.plot(ax=ax3, color="lightgray")
sc3 = ax3.scatter(
    df_kp2["lon"],
    df_kp2["lat"],
    c='black',
    s=5
)
ax3.set_title("kP=2 Locations")
ax3.set_xlabel("Longitude")
ax3.set_ylabel("Latitude")

world.plot(ax=ax4, color="lightgray")
sc4 = ax4.scatter(
    df_kp3["lon"],
    df_kp3["lat"],
    c='black',
    s=5
)
ax4.set_title("kP=3 Locations")
ax4.set_xlabel("Longitude")
ax4.set_ylabel("Latitude")

world.plot(ax=ax5, color="lightgray")
sc5 = ax5.scatter(
    df_kp4["lon"],
    df_kp4["lat"],
    c='black',
    s=5
)
ax5.set_title("kP=4 Locations")
ax5.set_xlabel("Longitude")
ax5.set_ylabel("Latitude")

world.plot(ax=ax6, color="lightgray")
sc6 = ax6.scatter(
    df_kp5["lon"],
    df_kp5["lat"],
    c='black',
    s=5
)
ax6.set_title("kP=5 Locations")
ax6.set_xlabel("Longitude")
ax6.set_ylabel("Latitude")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15,5.25), constrained_layout=True)
axes = axes.flatten()


for i in range(6):
    subset = floored[floored["kp"] == i]

    world.plot(ax=axes[i], color="lightgrey")
    subset.plot(ax=axes[i],
                markersize=1,
                column="xray0_ps",
                alpha=0.75,
                cmap=COLOR,
                legend=True)
    axes[i].set_title(f"Kp between {i} and {i+1}")
    axes[i].set_xlabel(f"Longitude")
    axes[i].set_ylabel(f"Latitude")

fig.suptitle("xray0_ps binned by kp", fontsize="xx-large")
plt.show()

**Analysis of X-ray binned by kp**

There are high and low values in all Kp bins. I expected the values to be higher in the higher bins, but that doesn't appear to be the case. I will investigate this further.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad[geo_rad["kp"] >= 5].plot(ax=ax,
                                 column="xray0_ps",
                                 markersize=20,
                                 alpha=0.75,
                                 cmap=COLOR,
                                 legend=True
)
ax.set_title("Kp >= 5")
plt.show()

Highlight Kp over 5 bcause it's when magnetic storms start.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad[geo_rad["kp"] < 3].plot(ax=ax,
                                 column="xray0_ps",
                                 markersize=20,
                                 alpha=0.75,
                                 cmap=COLOR,
                                 legend=True
)
ax.set_title("Kp < 3")
plt.show()

Highlight Kp less than 3 because that's when the atmosphere is considered quiet.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad[geo_rad["xray0_ps"] >= 10000].plot(ax=ax,
                                           column="kp",
                                           markersize=20,
                                           alpha=0.75,
                                           cmap=COLOR,
                                           legend=True,
                                           legend_kwds={"label": "Kp index"},
                                           vmax = KP_MAX,
                                           vmin = KP_MIN
)
ax.set_aspect("equal", adjustable="box")
ax.set_title("All xrays per second >= 10000 colored by Kp")
ax.set_xlabel(f"Longitude")
ax.set_ylabel(f"Latitude")
plt.show()

View all **xray0_ps >= 10,000** colored by Kp index.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad[geo_rad["xray0_ps"] <= 1].plot(ax=ax,
                                          column="kp",
                                          markersize=20,
                                          alpha=0.75,
                                          cmap=COLOR,
                                          legend=True,
                                          legend_kwds={"label": "Kp index"},
                                          vmax = KP_MAX,
                                          vmin = KP_MIN
)
ax.set_aspect("equal", adjustable="box")
ax.set_title("All xrays per second <= 1 colored by Kp")
ax.set_xlabel(f"Longitude")
ax.set_ylabel(f"Latitude")
plt.show()

View all **xray0_ps <= 1** colored by Kp index.

In [ ]:
for lag in ["kp_lag3", "kp_lag6", "kp_lag12", "kp_lag24", "kp_lag48", "kp_lag72"]:
    fig, axes = plt.subplots(2, 3, figsize=(14,5), constrained_layout=True)
    axes = axes.flatten()

    for i in range(6):
        subset = floored[floored[lag] == i]
        
        world.plot(ax=axes[i], color="lightgrey")
        subset.plot(ax=axes[i],
                    markersize=1,
                    column="xray0_ps",
                    alpha=0.75,
                    cmap=COLOR,
                    legend=True)
        axes[i].set_title(f"Kp between {i} and {i+1}")
        axes[i].set_aspect("equal", adjustable="box")
        axes[i].set_xlabel(f"Longitude")
        axes[i].set_ylabel(f"Latitude")
    fig.suptitle(f"xray0_ps binned by {lag}", fontsize="xx-large")
    plt.show()

#### Proton Radiation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14,5), constrained_layout=True)
axes = axes.flatten()


for i in range(6):
    subset = floored[floored["kp"] == i]
    
    world.plot(ax=axes[i], color="lightgrey")
    subset.plot(ax=axes[i],
                markersize=1,
                column="proton0_ps",
                alpha=0.75,
                cmap=COLOR,
                legend=True)
    axes[i].set_title(f"Kp between {i} and {i+1}")

fig.suptitle("proton0_ps binned by kp", fontsize="xx-large")
plt.show()

**Analysis of X-ray binned by kp**

Just like with X-rays, there are high and low values in all Kp bins. From my minimal understanding of proton vs X-ray radiation, this is less surprising, but it is still worth checking.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad[geo_rad["proton0_ps"] >= 10000].plot(ax=ax,
                                           column="kp",
                                           markersize=20,
                                           alpha=0.75,
                                           cmap=COLOR,
                                           legend=True,
                                           legend_kwds={"label": "Kp index"},
                                           vmax = KP_MAX,
                                           vmin = KP_MIN
)
ax.set_aspect("equal", adjustable="box")
ax.set_title("All protons per second >= 10000 colored by Kp")
plt.show()

View all **proton0_ps >= 10,000** colored by Kp index.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad[geo_rad["proton0_ps"] <= 1].plot(ax=ax,
                                          column="kp",
                                          markersize=20,
                                          alpha=0.75,
                                          cmap=COLOR,
                                          legend=True,
                                          legend_kwds={"label": "Kp index"},
                                          vmax = KP_MAX,
                                          vmin = KP_MIN
)
ax.set_aspect("equal", adjustable="box")
ax.set_title("All protons per second <= 1 colored by Kp")
plt.show()

View all **proton0_ps <= 2,000** colored by Kp index.

In [ ]:
for lag in ["kp_lag3", "kp_lag6", "kp_lag24", "kp_lag48"]:
    fig, axes = plt.subplots(2, 3, figsize=(14,5), constrained_layout=True)
    axes = axes.flatten()

    for i in range(6):
        subset = floored[floored[lag] == i]
        
        world.plot(ax=axes[i], color="lightgrey")
        subset.plot(ax=axes[i],
                    markersize=1,
                    column="proton0_ps",
                    alpha=0.75,
                    cmap=COLOR,
                    legend=True)
        axes[i].set_title(f"Kp between {i} and {i+1}")
    fig.suptitle(f"proton0_ps binned by {lag}", fontsize="xx-large")
    plt.show()

#### Electron Radiation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14,5), constrained_layout=True)
axes = axes.flatten()


for i in range(6):
    subset = floored[floored["kp"] == i]

    world.plot(ax=axes[i], color="lightgrey")
    subset.plot(ax=axes[i],
                markersize=1,
                column="electron0_ps",
                alpha=0.75,
                cmap=COLOR,
                legend=True)
    axes[i].set_title(f"Kp between {i} and {i+1}")

fig.suptitle("electron0_ps binned by kp", fontsize="xx-large")
plt.show()

**Analysis of electrons binned by kp**

Electrons tend to say pretty low across the board. There is no specific bin where it is consistently elevated. We have elevated values in all bins, but most values in all bins are low.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad[geo_rad["electron0_ps"] >= 10000].plot(ax=ax,
                                          column="kp",
                                          markersize=20,
                                          alpha=0.75,
                                          cmap=COLOR,
                                          legend=True,
                                          legend_kwds={"label": "Kp index"},
                                          vmax = KP_MAX,
                                          vmin = KP_MIN
)
ax.set_aspect("equal", adjustable="box")
ax.set_title("All electrons per second >= 10000 colored by Kp")
plt.show()

View all **electron0_ps >= 10,000** colored by Kp index.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

world.plot(ax=ax, color="lightgrey")
geo_rad[geo_rad["electron0_ps"] <= 1].plot(ax=ax,
                                          column="kp",
                                          markersize=20,
                                          alpha=0.75,
                                          cmap=COLOR,
                                          legend=True,
                                          legend_kwds={"label": "Kp index"},
                                          vmax = KP_MAX,
                                          vmin = KP_MIN
)
ax.set_aspect("equal", adjustable="box")
ax.set_title("All electrons per second <= 1 colored by Kp")
plt.show()

View all **electron0_ps <= 2,000** colored by Kp index.

In [ ]:
for lag in ["kp_lag3", "kp_lag6", "kp_lag24", "kp_lag48"]:
    fig, axes = plt.subplots(2, 3, figsize=(14,5), constrained_layout=True)
    axes = axes.flatten()

    for i in range(6):
        subset = floored[floored[lag] == i]
        
        world.plot(ax=axes[i], color="lightgrey")
        subset.plot(ax=axes[i],
                    markersize=1,
                    column="electron0_ps",
                    alpha=0.75,
                    cmap=COLOR,
                    legend=True)
        axes[i].set_title(f"Kp between {i} and {i+1}")
    fig.suptitle(f"electron0_ps binned by {lag}", fontsize="xx-large")
    plt.show()

### Box and Whisker Plot Exploration

In [ ]:
plt.figure(figsize=(14, 6))

plt.boxplot([
    df_kp0['xray0_ps'],
    df_kp1['xray0_ps'],
    df_kp2['xray0_ps'],
    df_kp3['xray0_ps'],
    df_kp4['xray0_ps'],
    df_kp5['xray0_ps']
])

plt.xticks(
    [1, 2, 3, 4, 5, 6],
    ['kP 0','kP 1','kP 2','kP 3','kP 4','kP 5']
)

plt.yscale('log')
plt.ylabel('Count per Second (Log Scale)')
plt.title('X-Ray Radiation Distributions (per second)')
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))

plt.boxplot([
    df_kp0['electron0_ps'],
    df_kp1['electron0_ps'],
    df_kp2['electron0_ps'],
    df_kp3['electron0_ps'],
    df_kp4['electron0_ps'],
    df_kp5['electron0_ps']
])

plt.xticks(
    [1, 2, 3, 4, 5, 6],
    ['kP 0','kP 1','kP 2','kP 3','kP 4','kP 5']
)

plt.yscale('log')
plt.ylabel('Count per Second (Log Scale)')
plt.title('Electron Radiation Distributions (per second)')
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))

plt.boxplot([
    df_kp0['proton0_ps'],
    df_kp1['proton0_ps'],
    df_kp2['proton0_ps'],
    df_kp3['proton0_ps'],
    df_kp4['proton0_ps'],
    df_kp5['proton0_ps']
])

plt.xticks(
    [1, 2, 3, 4, 5, 6],
    ['kP 0','kP 1','kP 2','kP 3','kP 4','kP 5']
)

plt.yscale('log')
plt.ylabel('Count per Second (Log Scale)')
plt.title('Proton Radiation Distributions (per second)')
plt.show()

To finish up the preliminary work, I wanted to look at the distributions of the x-ray radiation per second, electron radiation per second, and electron radiation per second across the 6 bins. 

**X-Ray Radiation Distribution:**

- Kp 2, 3, and 4 appears to have a similar distribution higher than the other 3 bins
- Kp 0 and 1 appears to have a similar distribution
- Kp 5 appears to have the lowest distribution for x-ray radiation

**Electron Radiation Distribution:**

- Kp 0 distribution appears to be slightly higher than the rest of the bins
- Kp 1, 2, 3, and 4 all appear to have a fairly similiar distribution
- Kp 5 distribution is significanly lower than the rest

**Proton Radiation Distribution:**

- Kp 2, 3, and 4 appears to have a similar distribution higher than the other 3 bins
- Kp 0 and 1 appears to have a similar distribution
- Kp 5 distribution is significanly lower than the rest

## Statistical Tests

### Linear Regression t-Tests

I wanted to run some linear regression t-test on our Kp values versus our radiation columns to see if there is a large or significant correlation there.

**Reminder:**

In a linear regression t-test, our hypotheses are...

- $H_0: \beta = 0$
- $H_a: \beta \neq 0$

then we will reject the null hypothesis if our p-value is less than 0.05

In [ ]:
def lin_reg_ttest(x,y,d):
    for i in x:
        for j in y:
            result = linregress(d[i],d[j])
            slope = result.slope
            intercept = result.intercept
            r_squared = result.rvalue**2
            p_value = result.pvalue
            t_stat = result.slope / result.stderr
            
            y_fit = slope * d[i] + intercept
            
            plt.figure(figsize=(10, 6))
            plt.scatter(d[i], d[j], s=5, alpha=0.5)
            plt.plot(d[i], y_fit, color="red")
            
            plt.xlabel(i)
            plt.ylabel(j)
            plt.title(i + ' vs ' + j)
            
            plt.show()
            
            print(f"Slope: {slope:.6f}")
            print(f"Intercept: {intercept:.6f}")
            print(f"R²: {r_squared:.4f}")
            print(f"t-statistic: {t_stat:.4f}")
            print(f"p-value: {p_value:.6g}")

            decision = "Reject Null" if p_value < 0.05 else "Fail to Reject"
            print(f"Decision: {decision}")

#### No Lag

In [ ]:
xlabs1 = ['kp']
ylabs1 = ['xray0_ps','electron0_ps','proton0_ps']

In [ ]:
lin_reg_ttest(xlabs1,ylabs1,df)

**Kp vs X-Ray 0**

The linear regression t-test between Kp and x-ray 0 resulted in a p-value that was extremely small, which causes us to reject the null hypothesis and find that there is a positive correlation between Kp and amount of x-ray radiaition. However, we also want to look at the $R^2$ value, which is also extremely small. This means that the linear regression line does not explain very much of the variation in the x-ray radiaition. Therefore, while the positive correlation is statistically significant, it also does not account for much of the variation.

**Kp vs Electron 0**

In the linear regression t-test between Kp and electron 0, we got an extremely large p-value. This means that we fail to reject the null hypothesis and conclude that there is not a relationship between Kp value and the amount of electron radiaition. 

**Kp vs Proton 0**

Finally, in the linear regression t-test between Kp and proton 0, we got a small p-value less than 0.05. Therefore, we will reject the null hypothesis and conclude that there is a positive relationship between Kp value and proton radiation. However, like in the Kp vs x-ray radiation test, the $R^2$ value is extremely small meaning that the linear regression line does not account for much of the variation in proton radiation. 

#### 3 Hour Lag

In [ ]:
xlabs3 = ['kp_lag3']
ylabs3 = ['xray0_ps','electron0_ps','proton0_ps']

In [ ]:
lin_reg_ttest(xlabs3,ylabs3,df)

#### 6 Hour Lag

In [ ]:
xlabs6 = ['kp_lag6']
ylabs6 = ['xray0_ps','electron0_ps','proton0_ps']

In [ ]:
lin_reg_ttest(xlabs6,ylabs6,df)

#### 12 Hour Lag

In [ ]:
xlabs12 = ['kp_lag12']
ylabs12 = ['xray0_ps','electron0_ps','proton0_ps']

In [ ]:
lin_reg_ttest(xlabs12,ylabs12,df)

#### 24 Hour Lag

In [ ]:
xlabs24 = ['kp_lag24']
ylabs24 = ['xray0_ps','electron0_ps','proton0_ps']

In [ ]:
lin_reg_ttest(xlabs24,ylabs24,df)

#### 48 Hour Lag

In [ ]:
xlabs48 = ['kp_lag48']
ylabs48 = ['xray0_ps','electron0_ps','proton0_ps']

In [ ]:
lin_reg_ttest(xlabs48,ylabs48,df)

#### 72 Hour Lag

In [ ]:
xlabs72 = ['kp_lag72']
ylabs72 = ['xray0_ps','electron0_ps','proton0_ps']

In [ ]:
lin_reg_ttest(xlabs72,ylabs72,df)

#### Arctic vs Antarctic

In [ ]:
lin_reg_ttest(xlabs1,ylabs1,df_arctic)

In [ ]:
lin_reg_ttest(xlabs1,ylabs1,df_antarctic)

### Welsh Two-Sample t-Tests

Next we will use Welsh's Two-Sample t-Test to check to see how likely it is that the distributions across Kp bins are to be equal to each other.

**Reminder:**

In Welch's two-sample t-test, our hypotheses are...

- $H_0: \bar{x}_{1} = \bar{x}_{2}$
- $H_a: \bar{x}_{1} \neq \bar{x}_{2}$

then we will reject the null hypothesis if our p-value is less than 0.05.

In [ ]:
def welsh_ttest(groups, names):
    results = []
    for i in range(len(groups)):
        for j in range(i + 1, len(groups)):
            t_stat, p_value = ttest_ind(
                groups[i],
                groups[j],
                equal_var=False
            )

            results.append({
                "Value 1": names[i],
                "Value 2": names[j],
                "t-statistic": t_stat,
                "p-value": p_value,
                "Decision": "Reject Null" if p_value < 0.05 else "Fail to Reject"
            })

    results = pd.DataFrame(results)

    results["t-statistic"] = results["t-statistic"].round(4)
    results["p-value"] = results["p-value"].round(4)

    return results

Here are the 3 distinct lists of columns that I plan to use the Welsh two-sample t-test function on.

In [ ]:
xray_bins = [df_kp0['xray0_ps'],
             df_kp1['xray0_ps'],
             df_kp2['xray0_ps'],
             df_kp3['xray0_ps'],
             df_kp4['xray0_ps'],
             df_kp5['xray0_ps']
            ]

electron_bins = [df_kp0['electron0_ps'],
                 df_kp1['electron0_ps'],
                 df_kp2['electron0_ps'],
                 df_kp3['electron0_ps'],
                 df_kp4['electron0_ps'],
                 df_kp5['electron0_ps']
                ]

proton_bins = [df_kp0['proton0_ps'],
               df_kp1['proton0_ps'],
               df_kp2['proton0_ps'],
               df_kp3['proton0_ps'],
               df_kp4['proton0_ps'],
               df_kp5['proton0_ps']
              ]

values = ['kP=0',
          'kP=1',
          'kP=2',
          'kP=3',
          'kP=4',
          'kP=5',
         ]

In [ ]:
xray_test_results = welsh_ttest(xray_bins, values)
xray_test_results

In [ ]:
print("XRAY SIMILAR COLUMNS")
print(xray_test_results['Decision'].value_counts())

The Welsh two-sample t-test has shown us that there are 3 sets of Kp bins that are where we can't reject the null hypothesis of them being a result of the distribution. These sets are 

- 0 and 5
- 1 and 5
- 2 and 3

These results are a little different than I expected based on the box and whisker plot from before. I thought that 0 and 1; 2 and 4; 3 and 4 would all be similiar pairs, but none of them met the probability criteria. I also did not expect 5 to pair with any other bin, but I think that the high standard deviation from the low number of data points in the bin helped keep the probability high enough to meet the criteria. 

In [ ]:
electron_test_results = welsh_ttest(electron_bins, values)
electron_test_results

In [ ]:
print("ELECTRON SIMILAR COLUMNS")
print(electron_test_results['Decision'].value_counts())

The Welsh two-sample t-test has shown us that there are 5 sets of Kp bins that are where we can't reject the null hypothesis of them being a result of the distribution. These sets are 

- 0 and 2
- 0 and 4
- 1 and 2
- 1 and 3
- 2 and 4

In [ ]:
proton_test_results = welsh_ttest(proton_bins, values)
proton_test_results

In [ ]:
print("PROTON SIMILAR COLUMNS")
print(proton_test_results['Decision'].value_counts())

The Welsh two-sample t-test has shown us that there are 7 sets of Kp bins that are where we can't reject the null hypothesis of them being a result of the distribution. These sets are 

- 0 and 4
- 1 and 2
- 1 and 3
- 1 and 5
- 2 and 3
- 2 and 5
- 3 and 5

### Correlation Matrices

In [ ]:
corr = df[["kp", "xray0_ps", "proton0_ps", "electron0_ps"]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap=COLOR, vmin=-1, vmax=1, ax=ax)

ax.xaxis.tick_top()          # move tick marks + labels to the top
plt.show()

**Analysis of correlation matrix**

All radiation columns have a very low correlation with Kp. This is consistent with what I've observed from the previous analysis in this file.

While the number is incredibly low, it is interesting that `kp` and `electron0_ps` show an inverse correlation.

Also, `proton0_ps` and `electron0_ps` have a very high correlation with each other. This is unsurprising from what's been seen in previous analysis.

In [ ]:
corr = df[["kp", "Ap", "xray0_ps", "proton0_ps", "electron0_ps"]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap=COLOR, vmin=-1, vmax=1, ax=ax)

ax.xaxis.tick_top()          # move tick marks + labels to the top
plt.show()

In [ ]:
corr = df[["kp_lag3", "kp_lag6", "kp_lag12", "kp_lag24", "kp_lag48", "kp_lag72", "xray0_ps", "proton0_ps", "electron0_ps"]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, annot=True, cmap=COLOR, vmin=-1, vmax=1, ax=ax)

ax.xaxis.tick_top()          # move tick marks + labels to the top
ax.tick_params(axis='x', labelsize=11)
plt.show()

**Analysis of lagged kp correlation matrix**

Just as expecte, there is not a very high correlation between any of the lagged kp values and any of our particle sensors. The closest we get is a 0.14 correlation between `xray0_ps` and `kp_lag24`, and that is nowhere near high enough to conclude correlation. I believe it is safe to say that, at least for our data, there is no correlation between kp index and X-ray, proton, or electron radiation at this altutude.